In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/customer_support_tickets.csv"
)

sample = df.sample(
    n=20000,
    random_state=42
)

sample.to_csv(
    "llm_input_sample.csv",
    index=False
)

In [ ]:
!nvidia-smi

Sun Jun  7 13:45:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Stage 1: Pseudo-Label Generation

The Support Integrity Auditor must operate without manually annotated mismatch labels. To bootstrap supervision, we first estimate the intrinsic severity of each ticket independent of its human-assigned priority.

The first signal uses a Large Language Model (LLM) in a zero-shot setting. The model evaluates ticket subject, description, issue category and channel, then assigns a severity score on a 4-level scale:

- 0 → Low
- 1 → Medium
- 2 → High
- 3 → Critical

This severity estimate acts as a semantic understanding signal and forms the foundation of the pseudo-labeling pipeline.

In [ ]:
def severity_to_int(label):

    label = label.lower()

    if "critical" in label:
        return 3

    if "high" in label:
        return 2

    if "medium" in label:
        return 1

    return 0

In [ ]:
def build_prompt(
    subject,
    description,
    category,
    channel
):

    return f"""
You are a support ticket severity auditor.

Assign a severity score.

Severity scale:

0 = Low
1 = Medium
2 = High
3 = Critical

Guidelines:

3:
Security incidents,
fraud,
service outages,
major operational disruption.

2:
Major functionality failures,
API failures,
application crashes,
login failures,
payment issues.

1:
User-impacting issues with workarounds.

0:
Questions,
clarifications,
information requests,
minor requests.

Ticket Category:
{category}

Ticket Channel:
{channel}

Subject:
{subject}

Description:
{description}

Return ONLY ONE NUMBER.

Severity Score:
"""

In [ ]:
#for prompt2
import re

def predict_severity(
    subject,
    description,
    category,
    channel
):

    prompt = build_prompt(
        subject,
        description,
        category,
        channel
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    outputs = model.generate(
    **inputs,
    max_new_tokens=5,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # DEBUG
    # print(response)
    numbers = re.findall(
    r"\b([0-3])\b",
    response
)

    if len(numbers) > 0:
        return int(numbers[0])

    return None

In [ ]:
import pandas as pd
import os
from tqdm import tqdm

INPUT_FILE = "llm_input_sample.csv"
CHECKPOINT_FILE = "llm_scores_checkpoint.csv"
SAVE_EVERY = 100

# -------------------------
# Load full dataset
# -------------------------

df = pd.read_csv(INPUT_FILE)

# -------------------------
# Resume if checkpoint exists
# -------------------------

if os.path.exists(CHECKPOINT_FILE):

    checkpoint = pd.read_csv(CHECKPOINT_FILE)

    completed_ids = set(checkpoint["Ticket_ID"])

    remaining_df = df[
        ~df["Ticket_ID"].isin(completed_ids)
    ].copy()

    results = checkpoint.to_dict("records")

    print(f"Resuming...")
    print(f"Completed: {len(checkpoint)}")
    print(f"Remaining: {len(remaining_df)}")

else:

    remaining_df = df.copy()

    results = []

    print("Starting fresh...")

# -------------------------
# Inference Loop
# -------------------------

counter = 0

for _, row in tqdm(
    remaining_df.iterrows(),
    total=len(remaining_df)
):

    try:

        score = predict_severity(
            row["Ticket_Subject"],
            row["Ticket_Description"],
            row["Issue_Category"],
            row["Ticket_Channel"]
        )

    except Exception as e:

        print("ERROR:", e)

        score = None

    results.append({
        "Ticket_ID": row["Ticket_ID"],
        "llm_score": score
    })

    counter += 1

    # Save checkpoint
    if counter % SAVE_EVERY == 0:

        pd.DataFrame(results).to_csv(
            CHECKPOINT_FILE,
            index=False
        )

# -------------------------
# Final Save
# -------------------------

final_df = pd.DataFrame(results)

final_df.to_csv(
    "llm_scores.csv",
    index=False
)

print("\nDone!")
print(final_df.shape)

Starting fresh...


100%|██████████| 20000/20000 [1:37:40<00:00,  3.41it/s]


Done!
(20000, 2)


In [ ]:
print(
    final_df["llm_score"]
    .value_counts()
)

llm_score
0    12464
2     4870
1     1565
3     1101
Name: count, dtype: int64


The generated llm.csv is further used for stage-1 pseudo label generation locally. Note: While trying to regenerate the metrices the dataset in the processed dataset should be used  as generating pseudo labels each time result in different pseudo labels on which the training was not done.